In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 05 · Risk Consolidation — Trust & Safety
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Consolidar las evaluaciones individuales de las reglas
# MAGIC Trust & Safety en una evaluación única por Review.
# MAGIC
# MAGIC ## Entrada
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_evaluation`
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_features`
# MAGIC
# MAGIC ## Salida
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_risk`
# MAGIC
# MAGIC ## Grano
# MAGIC
# MAGIC **1 Review × 1 Evaluation Run**
# MAGIC
# MAGIC ## Importante
# MAGIC
# MAGIC `weighted_risk_score` es un score heurístico.
# MAGIC
# MAGIC NO representa una probabilidad estadística de fraude
# MAGIC ni confirma que una Review sea fraudulenta.


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F

import uuid


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOV_SCHEMA = "yelp_gov"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"


EVALUATION_TABLE = (
    f"{SILVER}.slv_review_evaluation"
)

FEATURE_TABLE = (
    f"{SILVER}.slv_review_features"
)

RISK_TABLE = (
    f"{SILVER}.slv_review_risk"
)

RUN_TABLE = (
    f"{GOV}.evaluation_run"
)


PIPELINE_NAME = (
    "05_risk_consolidation"
)

RISK_MODEL_VERSION = "1.0.0"


# ------------------------------------------------------------
# OPCIONAL
#
# None:
# utiliza automáticamente el último Evaluation Run SUCCESS.
#
# También puedes colocar manualmente:
#
# EVALUATION_RUN_ID_OVERRIDE = "xxxx-xxxx-xxxx"
# ------------------------------------------------------------

EVALUATION_RUN_ID_OVERRIDE = None


# ------------------------------------------------------------
# THRESHOLDS HEURÍSTICOS
#
# Estos valores NO representan probabilidades estadísticas.
# Son criterios iniciales de priorización.
# ------------------------------------------------------------

LOW_THRESHOLD = 0.15
MEDIUM_THRESHOLD = 0.35
HIGH_THRESHOLD = 0.60


# ------------------------------------------------------------
# Investigación
# ------------------------------------------------------------

MIN_SIGNALS_INVESTIGATION = 2

MIN_SCORE_INVESTIGATION = 0.35


# ------------------------------------------------------------
# Prioridad P1
# ------------------------------------------------------------

P1_MIN_SIGNALS = 3
P1_MIN_SCORE = 0.60


RISK_CONSOLIDATION_ID = str(
    uuid.uuid4()
)


print("=" * 72)
print("YELPCONNECT - RISK CONSOLIDATION")
print("=" * 72)

print(
    f"Risk Consolidation ID : "
    f"{RISK_CONSOLIDATION_ID}"
)

print(
    f"Risk Model Version    : "
    f"{RISK_MODEL_VERSION}"
)

print("=" * 72)


# COMMAND ----------

# ============================================================
# 3. VALIDAR TABLAS REQUERIDAS
# ============================================================

required_tables = [

    EVALUATION_TABLE,
    FEATURE_TABLE,
    RUN_TABLE
]


for table_name in required_tables:

    if not spark.catalog.tableExists(
        table_name
    ):

        raise Exception(
            f"""
            ❌ Tabla requerida no encontrada:

            {table_name}
            """
        )


print(
    "✔ Tablas requeridas disponibles."
)


# COMMAND ----------

# ============================================================
# 4. DETERMINAR EVALUATION RUN
# ============================================================

if EVALUATION_RUN_ID_OVERRIDE:

    EVALUATION_RUN_ID = (
        EVALUATION_RUN_ID_OVERRIDE
    )

    run_exists = (

        spark.table(
            RUN_TABLE
        )

        .filter(
            F.col(
                "evaluation_run_id"
            )
            ==
            EVALUATION_RUN_ID
        )

        .filter(
            F.col(
                "execution_status"
            )
            ==
            "SUCCESS"
        )

        .limit(1)

        .count()
    )


    if run_exists == 0:

        raise Exception(
            f"""
            ❌ Evaluation Run no encontrado
            o no tiene estado SUCCESS:

            {EVALUATION_RUN_ID}
            """
        )

else:

    latest_run = (

        spark.table(
            RUN_TABLE
        )

        .filter(
            F.col(
                "execution_status"
            )
            ==
            "SUCCESS"
        )

        .orderBy(
            F.desc(
                "started_at"
            )
        )

        .select(
            "evaluation_run_id"
        )

        .first()
    )


    if latest_run is None:

        raise Exception(
            """
            ❌ No existe un Evaluation Run
            con estado SUCCESS.
            """
        )


    EVALUATION_RUN_ID = (
        latest_run.evaluation_run_id
    )


print(
    f"✔ Evaluation Run seleccionado: "
    f"{EVALUATION_RUN_ID}"
)


# COMMAND ----------

# ============================================================
# 5. CARGAR EVALUACIONES DEL RUN
# ============================================================

evaluations = (

    spark.table(
        EVALUATION_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )
)


evaluation_count = (
    evaluations.count()
)


review_count = (

    evaluations

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


expected_rule_count = (

    evaluations

    .select(
        "rule_version_id"
    )

    .distinct()

    .count()
)


print(
    f"Evaluaciones     : {evaluation_count:,}"
)

print(
    f"Reviews          : {review_count:,}"
)

print(
    f"Reglas del Run   : {expected_rule_count}"
)


if evaluation_count == 0:

    raise Exception(
        """
        ❌ El Evaluation Run seleccionado
        no contiene evaluaciones.
        """
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # Validación del grano
# MAGIC
# MAGIC Antes de consolidar verificamos que cada Review
# MAGIC tenga exactamente una evaluación por regla/version.


# COMMAND ----------

# ============================================================
# 6. VALIDAR REVIEW × REGLAS
# ============================================================

grain_validation = (

    evaluations

    .groupBy(
        "review_id"
    )

    .agg(

        F.count("*")
        .alias(
            "evaluation_count"
        ),

        F.countDistinct(
            "rule_version_id"
        )
        .alias(
            "distinct_rule_count"
        )
    )
)


invalid_reviews = (

    grain_validation

    .filter(

        (
            F.col(
                "evaluation_count"
            )
            !=
            expected_rule_count
        )

        |

        (
            F.col(
                "distinct_rule_count"
            )
            !=
            expected_rule_count
        )
    )

    .count()
)


if invalid_reviews > 0:

    raise Exception(
        f"""
        ❌ Se encontraron {invalid_reviews:,}
        Reviews con evaluaciones duplicadas
        o reglas faltantes.

        No se realizará la consolidación.
        """
    )


print(
    "✔ Grano Review × Rule validado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Consolidación de señales


# COMMAND ----------

# ============================================================
# 7. CONSOLIDAR POR REVIEW
# ============================================================

risk_base = (

    evaluations

    .groupBy(
        "review_id"
    )

    .agg(

        # ----------------------------------------------------
        # Evaluation Run ID (constant — already filtered)
        # ----------------------------------------------------

        F.first(
            "evaluation_run_id"
        )
        .alias(
            "evaluation_run_id"
        ),


        # ----------------------------------------------------
        # Cantidad total de reglas
        # ----------------------------------------------------

        F.count("*")
        .alias(
            "rule_count"
        ),


        # ----------------------------------------------------
        # SIGNAL
        # ----------------------------------------------------

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "SIGNAL",

                F.lit(1)
            )

            .otherwise(
                F.lit(0)
            )
        )
        .cast("int")
        .alias(
            "signal_count"
        ),


        # ----------------------------------------------------
        # NO SIGNAL
        # ----------------------------------------------------

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NO_SIGNAL",

                F.lit(1)
            )

            .otherwise(
                F.lit(0)
            )
        )
        .cast("int")
        .alias(
            "no_signal_count"
        ),


        # ----------------------------------------------------
        # NOT EVALUABLE
        # ----------------------------------------------------

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NOT_EVALUABLE",

                F.lit(1)
            )

            .otherwise(
                F.lit(0)
            )
        )
        .cast("int")
        .alias(
            "not_evaluable_count"
        ),


        # ----------------------------------------------------
        # SCORE
        #
        # Se suma el peso que ya asignó cada regla.
        # ----------------------------------------------------

        F.round(

            F.sum(
                "risk_score"
            ),

            4
        )
        .alias(
            "weighted_risk_score"
        ),


        # ----------------------------------------------------
        # Reglas que generaron SIGNAL
        # ----------------------------------------------------

        F.sort_array(

            F.collect_set(

                F.when(

                    F.col(
                        "evaluation_result"
                    )
                    ==
                    "SIGNAL",

                    F.col(
                        "rule_id"
                    )
                )
            )
        )
        .alias(
            "signal_rules"
        ),


        # ----------------------------------------------------
        # Versiones activadas
        # ----------------------------------------------------

        F.sort_array(

            F.collect_set(

                F.when(

                    F.col(
                        "evaluation_result"
                    )
                    ==
                    "SIGNAL",

                    F.col(
                        "rule_version_id"
                    )
                )
            )
        )
        .alias(
            "signal_rule_versions"
        ),


        # ----------------------------------------------------
        # Reglas no evaluables
        # ----------------------------------------------------

        F.sort_array(

            F.collect_set(

                F.when(

                    F.col(
                        "evaluation_result"
                    )
                    ==
                    "NOT_EVALUABLE",

                    F.col(
                        "rule_id"
                    )
                )
            )
        )
        .alias(
            "not_evaluable_rules"
        )
    )
)


# COMMAND ----------

# ============================================================
# 8. COBERTURA DE EVALUACIÓN
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "evaluable_rule_count",

        F.col(
            "rule_count"
        )

        -

        F.col(
            "not_evaluable_count"
        )
    )

    .withColumn(
        "evaluation_coverage_pct",

        F.round(

            (
                F.col(
                    "evaluable_rule_count"
                )

                /

                F.col(
                    "rule_count"
                )
            )

            *
            100,

            2
        )
    )

    .withColumn(
        "has_incomplete_evaluation",

        F.col(
            "not_evaluable_count"
        ) > 0
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Nivel de riesgo heurístico


# COMMAND ----------

# ============================================================
# 9. SIGNAL LEVEL
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "signal_level",

        F.when(
            F.col(
                "weighted_risk_score"
            )
            >=
            HIGH_THRESHOLD,

            F.lit(
                "HIGH"
            )
        )

        .when(
            F.col(
                "weighted_risk_score"
            )
            >=
            MEDIUM_THRESHOLD,

            F.lit(
                "MEDIUM"
            )
        )

        .when(
            F.col(
                "weighted_risk_score"
            )
            >=
            LOW_THRESHOLD,

            F.lit(
                "LOW"
            )
        )

        .otherwise(
            F.lit(
                "NONE"
            )
        )
    )
)


# COMMAND ----------

# ============================================================
# 10. REQUIRES INVESTIGATION
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "requires_investigation",

        (
            F.col(
                "signal_count"
            )
            >=
            MIN_SIGNALS_INVESTIGATION
        )

        |

        (
            F.col(
                "weighted_risk_score"
            )
            >=
            MIN_SCORE_INVESTIGATION
        )
    )
)


# COMMAND ----------

# ============================================================
# 11. PRIORIDAD DE INVESTIGACIÓN
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "investigation_priority",

        # ----------------------------------------------------
        # P1
        # ----------------------------------------------------

        F.when(

            (
                F.col(
                    "signal_count"
                )
                >=
                P1_MIN_SIGNALS
            )

            |

            (
                F.col(
                    "weighted_risk_score"
                )
                >=
                P1_MIN_SCORE
            ),

            F.lit(
                "P1"
            )
        )


        # ----------------------------------------------------
        # P2
        # ----------------------------------------------------

        .when(

            (
                F.col(
                    "signal_count"
                )
                >=
                MIN_SIGNALS_INVESTIGATION
            )

            |

            (
                F.col(
                    "weighted_risk_score"
                )
                >=
                MIN_SCORE_INVESTIGATION
            ),

            F.lit(
                "P2"
            )
        )


        # ----------------------------------------------------
        # P3
        # ----------------------------------------------------

        .when(

            F.col(
                "signal_count"
            )
            >=
            1,

            F.lit(
                "P3"
            )
        )


        # ----------------------------------------------------
        # NONE
        # ----------------------------------------------------

        .otherwise(
            F.lit(
                "NONE"
            )
        )
    )
)


# COMMAND ----------

# ============================================================
# 12. CLASIFICACIÓN FUNCIONAL
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "risk_classification",

        F.when(

            F.col(
                "requires_investigation"
            )
            ==
            True,

            F.lit(
                "POTENTIALLY_SUSPICIOUS"
            )
        )

        .when(

            F.col(
                "signal_count"
            )
            >
            0,

            F.lit(
                "SIGNAL_DETECTED"
            )
        )

        .otherwise(
            F.lit(
                "NO_RELEVANT_SIGNAL"
            )
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Agregar contexto de la Review


# COMMAND ----------

# ============================================================
# 13. FEATURE SNAPSHOT
# ============================================================

feature_context = (

    spark.table(
        FEATURE_TABLE
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "stars",
        "review_ts",

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",

        "user_review_count",

        "user_business_review_count",
        "user_business_concentration",

        "text_length",
        "text_repeat_count",

        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_spike_ratio",

        "feature_execution_id"
    )
)


# COMMAND ----------

# ============================================================
# 14. RISK + CONTEXTO
# ============================================================

review_risk = (

    risk_base

    .join(
        feature_context,
        "review_id",
        "left"
    )

    .withColumn(
        "risk_consolidation_id",

        F.lit(
            RISK_CONSOLIDATION_ID
        )
    )

    .withColumn(
        "risk_model_version",

        F.lit(
            RISK_MODEL_VERSION
        )
    )

    .withColumn(
        "risk_processed_at",

        F.current_timestamp()
    )

    .withColumn(
        "risk_pipeline_name",

        F.lit(
            PIPELINE_NAME
        )
    )
)


# COMMAND ----------

# ============================================================
# 15. SELECCIÓN FINAL
# ============================================================

review_risk = (

    review_risk

    .select(

        # ----------------------------------------------------
        # Identificación
        # ----------------------------------------------------

        "review_id",
        "user_id",
        "business_id",

        "evaluation_run_id",


        # ----------------------------------------------------
        # Review
        # ----------------------------------------------------

        "review_ts",
        "stars",


        # ----------------------------------------------------
        # Consolidación de reglas
        # ----------------------------------------------------

        "rule_count",
        "evaluable_rule_count",
        "signal_count",
        "no_signal_count",
        "not_evaluable_count",

        "signal_rules",
        "signal_rule_versions",
        "not_evaluable_rules",


        # ----------------------------------------------------
        # Riesgo
        # ----------------------------------------------------

        "weighted_risk_score",
        "signal_level",

        "requires_investigation",
        "investigation_priority",

        "risk_classification",


        # ----------------------------------------------------
        # Cobertura / explicabilidad
        # ----------------------------------------------------

        "evaluation_coverage_pct",
        "has_incomplete_evaluation",


        # ----------------------------------------------------
        # Features principales
        # ----------------------------------------------------

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",

        "user_review_count",

        "user_business_review_count",
        "user_business_concentration",

        "text_length",
        "text_repeat_count",

        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_spike_ratio",


        # ----------------------------------------------------
        # Trazabilidad
        # ----------------------------------------------------

        "feature_execution_id",

        "risk_consolidation_id",
        "risk_model_version",
        "risk_processed_at",
        "risk_pipeline_name"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Validación del grano final
# MAGIC
# MAGIC Debemos terminar con:
# MAGIC
# MAGIC **1 fila por Review para el Evaluation Run seleccionado.**


# COMMAND ----------

# ============================================================
# 16. VALIDAR GRANO FINAL
# ============================================================

risk_row_count = (
    review_risk.count()
)


risk_review_count = (

    review_risk

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


print(
    f"Reviews origen       : {review_count:,}"
)

print(
    f"Risk rows generadas  : {risk_row_count:,}"
)

print(
    f"Review ID únicos     : {risk_review_count:,}"
)


if (
    risk_row_count != review_count

    or

    risk_review_count != review_count
):

    raise Exception(
        """
        ❌ No se cumple el grano:

        1 Review × 1 Evaluation Run
        """
    )


print(
    "✔ Grano de Risk Consolidation validado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Persistencia Delta
# MAGIC
# MAGIC Se usa MERGE para hacer el proceso idempotente.
# MAGIC
# MAGIC Si se vuelve a ejecutar el mismo Evaluation Run,
# MAGIC no se duplican las Reviews.


# COMMAND ----------

# ============================================================
# 17. CREAR TABLA DESTINO
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {RISK_TABLE}
    (
        review_id                     STRING,
        user_id                       STRING,
        business_id                   STRING,

        evaluation_run_id             STRING,

        review_ts                     TIMESTAMP,
        stars                         DOUBLE,

        rule_count                    BIGINT,
        evaluable_rule_count          BIGINT,
        signal_count                  INT,
        no_signal_count               INT,
        not_evaluable_count           INT,

        signal_rules                  ARRAY<STRING>,
        signal_rule_versions          ARRAY<STRING>,
        not_evaluable_rules           ARRAY<STRING>,

        weighted_risk_score           DOUBLE,
        signal_level                  STRING,

        requires_investigation        BOOLEAN,
        investigation_priority        STRING,

        risk_classification           STRING,

        evaluation_coverage_pct       DOUBLE,
        has_incomplete_evaluation     BOOLEAN,

        rating_deviation              DOUBLE,
        account_age_days              INT,

        user_reviews_30m              BIGINT,
        user_reviews_1h               BIGINT,
        user_reviews_24h              BIGINT,

        user_review_count             BIGINT,

        user_business_review_count    BIGINT,
        user_business_concentration   DOUBLE,

        text_length                   INT,
        text_repeat_count             BIGINT,

        business_reviews_day          BIGINT,
        business_avg_daily_reviews    DOUBLE,
        business_spike_ratio          DOUBLE,

        feature_execution_id          STRING,

        risk_consolidation_id         STRING,
        risk_model_version            STRING,
        risk_processed_at             TIMESTAMP,
        risk_pipeline_name            STRING
    )
    USING DELTA
    """
)


# COMMAND ----------

# ============================================================
# 18. TEMP VIEW
# ============================================================

review_risk.createOrReplaceTempView(
    "src_review_risk"
)


# COMMAND ----------

# ============================================================
# 19. MERGE IDEMPOTENTE
# ============================================================

spark.sql(
    f"""
    MERGE INTO
        {RISK_TABLE} AS target

    USING
        src_review_risk AS source

    ON
        target.review_id =
            source.review_id

        AND

        target.evaluation_run_id =
            source.evaluation_run_id

    WHEN MATCHED THEN

        UPDATE SET *

    WHEN NOT MATCHED THEN

        INSERT *
    """
)


print(
    f"✔ Tabla actualizada: "
    f"{RISK_TABLE}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Validación de persistencia


# COMMAND ----------

# ============================================================
# 20. VALIDAR RUN PERSISTIDO
# ============================================================

persisted_count = (

    spark.table(
        RISK_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )

    .count()
)


print(
    f"Persistidas para Run : "
    f"{persisted_count:,}"
)


if persisted_count != review_count:

    raise Exception(
        """
        ❌ El número de Reviews persistidas
        no coincide con la ejecución origen.
        """
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # Distribución de niveles de riesgo


# COMMAND ----------

display(

    review_risk

    .groupBy(
        "signal_level"
    )

    .agg(

        F.count("*")
        .alias(
            "reviews"
        ),

        F.round(

            F.avg(
                "weighted_risk_score"
            ),

            4
        )
        .alias(
            "avg_score"
        )
    )

    .orderBy(

        F.when(
            F.col(
                "signal_level"
            )
            ==
            "HIGH",
            1
        )

        .when(
            F.col(
                "signal_level"
            )
            ==
            "MEDIUM",
            2
        )

        .when(
            F.col(
                "signal_level"
            )
            ==
            "LOW",
            3
        )

        .otherwise(
            4
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Distribución de prioridades


# COMMAND ----------

display(

    review_risk

    .groupBy(
        "investigation_priority"
    )

    .count()

    .orderBy(

        F.when(
            F.col(
                "investigation_priority"
            )
            ==
            "P1",
            1
        )

        .when(
            F.col(
                "investigation_priority"
            )
            ==
            "P2",
            2
        )

        .when(
            F.col(
                "investigation_priority"
            )
            ==
            "P3",
            3
        )

        .otherwise(
            4
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Casos prioritarios para investigación


# COMMAND ----------

display(

    review_risk

    .filter(
        F.col(
            "requires_investigation"
        )
        ==
        True
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "signal_count",
        "signal_rules",

        "weighted_risk_score",
        "signal_level",

        "investigation_priority",

        "evaluation_coverage_pct",

        "rating_deviation",
        "account_age_days",
        "user_reviews_30m",
        "user_business_concentration",
        "text_repeat_count",
        "business_spike_ratio"
    )

    .orderBy(

        F.when(
            F.col(
                "investigation_priority"
            )
            ==
            "P1",
            1
        )

        .otherwise(
            2
        ),

        F.desc(
            "weighted_risk_score"
        ),

        F.desc(
            "signal_count"
        )
    )

    .limit(100)
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Reviews con múltiples señales
# MAGIC
# MAGIC Estos casos son especialmente útiles para análisis,
# MAGIC porque combinan patrones independientes.


# COMMAND ----------

display(

    review_risk

    .filter(
        F.col(
            "signal_count"
        )
        >=
        2
    )

    .select(

        "review_id",
        "signal_count",
        "signal_rules",

        "weighted_risk_score",
        "signal_level",

        "investigation_priority"
    )

    .orderBy(

        F.desc(
            "signal_count"
        ),

        F.desc(
            "weighted_risk_score"
        )
    )

    .limit(100)
)


# COMMAND ----------

# ============================================================
# 21. KPIs DE EJECUCIÓN
# ============================================================

execution_kpis = (

    review_risk

    .agg(

        F.count("*")
        .alias(
            "reviews_evaluated"
        ),


        F.sum(

            F.when(
                F.col(
                    "signal_count"
                )
                >
                0,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_with_signal"
        ),


        F.sum(

            F.when(
                F.col(
                    "requires_investigation"
                )
                ==
                True,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_for_investigation"
        ),


        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P1",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p1"
        ),


        F.sum(

            F.when(
                F.col(
                    "has_incomplete_evaluation"
                )
                ==
                True,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_incomplete_evaluation"
        )
    )
)


display(
    execution_kpis
)


# COMMAND ----------

# ============================================================
# 22. RESUMEN FINAL
# ============================================================

kpis = (
    execution_kpis.first()
)


print()
print("=" * 72)
print("RISK CONSOLIDATION FINALIZADO")
print("=" * 72)

print(
    f"Evaluation Run            : "
    f"{EVALUATION_RUN_ID}"
)

print(
    f"Reviews evaluadas         : "
    f"{kpis['reviews_evaluated']:,}"
)

print(
    f"Reviews con señal         : "
    f"{kpis['reviews_with_signal']:,}"
)

print(
    f"Requieren investigación   : "
    f"{kpis['reviews_for_investigation']:,}"
)

print(
    f"Prioridad P1              : "
    f"{kpis['priority_p1']:,}"
)

print(
    f"Evaluación incompleta     : "
    f"{kpis['reviews_incomplete_evaluation']:,}"
)

print(
    f"Output                    : "
    f"{RISK_TABLE}"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done
# MAGIC
# MAGIC - [ ] Se seleccionó un Evaluation Run exitoso.
# MAGIC - [ ] Se validó el grano Review × Rule.
# MAGIC - [ ] Se consolidaron todas las reglas por Review.
# MAGIC - [ ] Se generó `signal_count`.
# MAGIC - [ ] Se generó `signal_rules`.
# MAGIC - [ ] Se generó `weighted_risk_score`.
# MAGIC - [ ] Se generó `signal_level`.
# MAGIC - [ ] Se generó `requires_investigation`.
# MAGIC - [ ] Se generó `investigation_priority`.
# MAGIC - [ ] Se conserva `NOT_EVALUABLE`.
# MAGIC - [ ] Se calculó cobertura de evaluación.
# MAGIC - [ ] El resultado es idempotente mediante MERGE.
# MAGIC - [ ] No se etiqueta fraude confirmado.